In [ ]:
# Installation
!pip install optuna

In [ ]:
# Building a classifier model to predict the class of the copper-binding site
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

# 1. Load Dataset
df = pd.read_csv("nonredundant_binding_sites_water_fixed.csv")

# 2. Numerical Features
numerical_cols = [
    'ARG_Count','ASN_Count','ASP_Count','CYS_Count','GLN_Count','GLU_Count',
    'GLY_Count','HIS_Count','MET_Count','SER_Count','THR_Count','TYR_Count',
    'VAL_Count','PHE_Count','ILE_Count','LEU_Count','ALA_Count','LYS_Count',
    'TRP_Count',
    'Atom_N','Atom_ND','Atom_NE','Atom_NH','Atom_NZ','Atom_O',
    'Atom_OD','Atom_OE','Atom_OG','Atom_OH','Atom_SD','Atom_SG',
    'Class_Nature_Non-polar',
    'Class_Nature_Polar acidic',
    'Class_Nature_Polar basic',
    'Class_Nature_Polar neutral',
    'Class_Nature_Polar O',
    'Aromatic_Count',
    'Average_Isoelectric_Point',
    'Average_Hydrophobicity'
]

df[numerical_cols] = df[numerical_cols].fillna(0)

# 3. Merge into 13 Classes
cluster_map = {
1:"T1_C1",2:"T1_C3",3:"T1_C2",
4:"T1_C4",5:"T1_C4",6:"T1_C4",
7:"T2_C1",8:"T2_C1",
9:"T2_C2",10:"T2_C2",11:"T2_C2",12:"T2_C2",13:"T2_C2",14:"T2_C2",15:"T2_C2",
16:"T2_C3",17:"T2_C3",
18:"T2_C4",19:"T2_C4",
20:"T2_C5",21:"T2_C5",
22:"T2_C6",23:"T2_C6",24:"T2_C6",
25:"T2_C7",26:"T2_C7",
27:"T2_C8",
28:"T2_C9",29:"T2_C9",30:"T2_C9"
}

df["Class13"] = df["Cluster_Label"].map(cluster_map)

if df["Class13"].isna().any():
    raise ValueError("Unmapped cluster labels.")

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["Class13"])
X = df[numerical_cols].values

# 4. Nested stratified kfold
OUTER_FOLDS = 4
INNER_FOLDS = 3
N_TRIALS = 50

outer_cv = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=42
)

inner_cv = StratifiedKFold(
    n_splits=INNER_FOLDS,
    shuffle=True,
    random_state=42
)

def build_pipeline(model):
    return Pipeline([
        ("scaler", MinMaxScaler()),
        ("model", model)
    ])

def objective(trial, model_name, X_train, y_train):

    if model_name == "LogisticRegression":
        model = LogisticRegression(
            C=trial.suggest_float("C",0.01,20,log=True),
            max_iter=3000,
            random_state=42
        )

    elif model_name == "KNN":
        model = KNeighborsClassifier(
            n_neighbors=trial.suggest_int("n_neighbors",3,25),
            weights=trial.suggest_categorical(
                "weights",
                ["uniform","distance"]
            )
        )

    elif model_name == "SVM":
        model = SVC(
            C=trial.suggest_float("C",0.1,20,log=True),
            gamma=trial.suggest_float("gamma",1e-3,1,log=True),
            kernel=trial.suggest_categorical(
                "kernel",
                ["rbf","poly"]
            )
        )

    elif model_name == "RandomForest":
        model = RandomForestClassifier(
            n_estimators=trial.suggest_int(
                "n_estimators",
                100,
                300
            ),
            max_depth=trial.suggest_int(
                "max_depth",
                5,
                20
            ),
            min_samples_split=trial.suggest_int(
                "min_samples_split",
                2,
                10
            ),
            min_samples_leaf=trial.suggest_int(
                "min_samples_leaf",
                1,
                5
            ),
            random_state=42
        )

    elif model_name == "XGBoost":
        model = XGBClassifier(
            n_estimators=trial.suggest_int(
                "n_estimators",
                100,
                300
            ),
            max_depth=trial.suggest_int(
                "max_depth",
                3,
                8
            ),
            learning_rate=trial.suggest_float(
                "learning_rate",
                0.02,
                0.20,
                log=True
            ),
            subsample=trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),
            colsample_bytree=trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            ),
            reg_alpha=trial.suggest_float(
                "reg_alpha",
                1e-4,
                1,
                log=True
            ),
            reg_lambda=trial.suggest_float(
                "reg_lambda",
                0.1,
                5,
                log=True
            ),
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=42
        )

    pipe = build_pipeline(model)

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=inner_cv,
        scoring="balanced_accuracy",
        n_jobs=-1
    )

    return scores.mean()

models = [
    "LogisticRegression",
    "KNN",
    "SVM",
    "RandomForest",
    "XGBoost"
]

all_results = []

for model_name in models:

    print(f"\n===== {model_name} =====")

    acc=[]
    bal=[]
    macro=[]

    best_params_all=[]
    # NEW
    all_true = []
    all_pred = []

    for fold,(train_idx,test_idx) in enumerate(outer_cv.split(X,y),1):

        X_train=X[train_idx]
        X_test=X[test_idx]

        y_train=y[train_idx]
        y_test=y[test_idx]

        study=optuna.create_study(
            direction="maximize",
            pruner=optuna.pruners.MedianPruner()
        )

        study.optimize(
            lambda trial: objective(
                trial,
                model_name,
                X_train,
                y_train
            ),
            n_trials=N_TRIALS,
            show_progress_bar=False
        )

        params=study.best_params
        best_params_all.append(params)

        if model_name=="LogisticRegression":
            model=LogisticRegression(
                **params,
                max_iter=3000,
                random_state=42
            )

        elif model_name=="KNN":
            model=KNeighborsClassifier(**params)

        elif model_name=="SVM":
            model=SVC(**params)

        elif model_name=="RandomForest":
            model=RandomForestClassifier(
                **params,
                random_state=42
            )

        else:
            model=XGBClassifier(
                **params,
                objective="multi:softprob",
                eval_metric="mlogloss",
                random_state=42
            )

        pipe=build_pipeline(model)

        pipe.fit(X_train,y_train)

        pred=pipe.predict(X_test)
        all_true.extend(y_test)
        all_pred.extend(pred)

        acc.append(
            accuracy_score(y_test,pred)
        )

        bal.append(
            balanced_accuracy_score(y_test,pred)
        )

        macro.append(
            f1_score(
                y_test,
                pred,
                average="macro"
            )
        )

        print(
            f"Fold {fold}: "
            f"BA={bal[-1]:.4f}"
        )

    print("\nResults")
    print(
        f"Accuracy : {np.mean(acc):.2f} ± {np.std(acc):.2f}"
    )
    print(
        f"Balanced : {np.mean(bal):.2f} ± {np.std(bal):.2f}"
    )
    print(
        f"Macro F1 : {np.mean(macro):.2f} ± {np.std(macro):.2f}"
    )

    # Confusion Matrix (%)
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns

    cm = confusion_matrix(
        all_true,
        all_pred,
        labels=np.arange(len(label_encoder.classes_))
    )

    cm_percent = np.divide(
        cm.astype(float),
        cm.sum(axis=1, keepdims=True),
        where=cm.sum(axis=1, keepdims=True) != 0
    ) * 100

    plt.figure(figsize=(10,8))

    sns.heatmap(
        cm_percent,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
        vmin=0,
        vmax=100
    )

    plt.xlabel("Predicted")

    plt.ylabel("True")

    plt.title(f"{model_name} Confusion Matrix (%)")

    plt.xticks(rotation=45)

    plt.yticks(rotation=0)

    plt.tight_layout()

    plt.savefig(
        f"{model_name}_ConfusionMatrix.png",
        dpi=300
    )

    plt.show()

    # Per-class Accuracy
    df_eval = pd.DataFrame({
        "True_Label": all_true,
        "Pred_Label": all_pred
    })

    df_eval["Correct"] = (
        df_eval["True_Label"] ==
        df_eval["Pred_Label"]
    )

    per_class = (
        df_eval
        .groupby("True_Label")["Correct"]
        .mean()
        *100
    )

    print("\nPer-class Accuracy (%)")

    print(per_class.sort_index())

    per_class.to_csv(

        f"{model_name}_PerClassAccuracy.csv"

    )

    prediction_df = pd.DataFrame({

          "True": label_encoder.inverse_transform(all_true),

          "Predicted": label_encoder.inverse_transform(all_pred)

      })

    prediction_df.to_csv(

          f"{model_name}_Predictions.csv",

          index=False

      )

    pd.DataFrame(best_params_all).to_csv(
        f"{model_name}_best_params.csv",
        index=False
    )

    all_results.append({
        "Model":model_name,
        "Accuracy Mean":np.mean(acc),
        "Accuracy SD":np.std(acc),
        "Balanced Mean":np.mean(bal),
        "Balanced SD":np.std(bal),
        "MacroF1 Mean":np.mean(macro),
        "MacroF1 SD":np.std(macro)
    })

summary=pd.DataFrame(all_results)
print(summary)
summary.to_csv("NestedCV_Results.csv",index=False)

In [ ]:
# FINAL SVM MODEL FOR DEPLOYMENT
import numpy as np
import pandas as pd
import joblib
import optuna
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score


# 1. Check the data
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")
print(pd.Series(y).value_counts().sort_index())

# 2. Define final SVM objective
final_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


def final_svm_objective(trial):

    svm = SVC(
        C=trial.suggest_float(
            "C",
            0.1,
            20,
            log=True
        ),

        gamma=trial.suggest_float(
            "gamma",
            1e-3,
            1,
            log=True
        ),

        kernel=trial.suggest_categorical(
            "kernel",
            ["rbf", "poly"]
        )
    )

    pipeline = Pipeline([
        ("scaler", MinMaxScaler()),
        ("model", svm)
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=final_cv,
        scoring="balanced_accuracy",
        n_jobs=-1
    )

    return scores.mean()

# 3. Run final Optuna optimization
FINAL_N_TRIALS = 50

final_study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner()
)

final_study.optimize(
    final_svm_objective,
    n_trials=FINAL_N_TRIALS,
    show_progress_bar=True
)


# 4. Display selected hyperparameters
print("\n========================================")
print("FINAL SVM OPTIMIZATION")
print("========================================")

print("\nBest balanced accuracy:")
print(f"{final_study.best_value:.4f}")

print("\nBest SVM parameters:")
print(final_study.best_params)


# 5. Create final scaler
final_scaler = MinMaxScaler()

X_scaled = final_scaler.fit_transform(X)


# 6. Create final SVM
final_svm = SVC(
    **final_study.best_params
)

# 7. Train final SVM on ALL available data
final_svm.fit(
    X_scaled,
    y
)

print("\n✅ Final SVM trained on ALL available data.")

# 8. Create label mapping
label_map = {
    int(i): str(label)
    for i, label in enumerate(label_encoder.classes_)
}

print("\nLabel mapping:")
print(label_map)

# 9. Save the three deployment files
joblib.dump(
    final_svm,
    "final_cluster_classifier.pkl"
)

joblib.dump(
    final_scaler,
    "feature_scaler.pkl"
)

joblib.dump(
    label_map,
    "label_mapping.pkl"
)

# 10. Verify that the files exist
import os

print("DEPLOYMENT FILES")
for filename in [
    "final_cluster_classifier.pkl",
    "feature_scaler.pkl",
    "label_mapping.pkl"
]:

    if os.path.exists(filename):

        size_kb = os.path.getsize(filename) / 1024

        print(
            f"✅ {filename} "
            f"({size_kb:.1f} KB)"
        )

    else:

        print(
            f"❌ {filename} NOT FOUND"
        )